# M25 — RanPAC ResNet-50 on CIFAR-100 (train-only)

This notebook runs the preregistered cross-backbone confirmation: six paired streams, frozen ImageNet ResNet-50 features, RanPAC random-ReLU width 10,000, and Exact/SRQ-INT8/SRQ-Adaptive. It never materializes `test.pt` and does not inspect test accuracy.

Enable a single Colab GPU before running all cells.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, urllib.request
from pathlib import Path
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_COMMIT = 'cb09e99'
WORK_DIR = '/content/SOHO-CL'
CONFIG = 'configs/srq_generalization_m25_ranpac_resnet_cifar_train_only.json'
CACHE_DIR = '/content/srq_m25_resnet_features'
OUTPUT_DIR = '/content/srq_m25_output'
CHECKPOINT = '/content/resnet50-11ad3fa6.pth'
if Path(WORK_DIR).exists(): shutil.rmtree(WORK_DIR)
subprocess.run(['git', 'clone', '--quiet', '--no-checkout', REPO_GIT_URL, WORK_DIR], check=True)
subprocess.run(['git', 'checkout', '--detach', '--quiet', REPO_COMMIT], cwd=WORK_DIR, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt'], cwd=WORK_DIR, check=True)
os.chdir(WORK_DIR)
import torch
assert torch.cuda.is_available(), 'Enable a Colab GPU and restart from this cell.'
print('GPU:', torch.cuda.get_device_name(0))
print('COMMIT:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
import tarfile
CIFAR_ROOT = Path('/content/cifar100')
CIFAR_DIR = CIFAR_ROOT / 'cifar-100-python'
if not (CIFAR_DIR / 'train').is_file():
    archive = Path('/content/cifar-100-python.tar.gz')
    if not archive.is_file():
        print('Downloading CIFAR-100 archive...', flush=True)
        subprocess.run([
            'wget', '--show-progress', '--timeout=60', '--tries=3',
            'https://www.cs.toronto.edu/~kriz/cifar-100-python.tar.gz',
            '-O', str(archive)
        ], check=True, timeout=300)
    print('Extracting CIFAR-100 archive...', flush=True)
    CIFAR_ROOT.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, 'r:gz') as handle:
        handle.extractall(CIFAR_ROOT)
assert (CIFAR_DIR / 'train').is_file(), 'CIFAR-100 train split is missing.'
print('CIFAR-100 train split ready:', CIFAR_DIR / 'train')

In [ ]:
checkpoint_url = 'https://download.pytorch.org/models/resnet50-11ad3fa6.pth'
if not Path(CHECKPOINT).is_file():
    urllib.request.urlretrieve(checkpoint_url, CHECKPOINT)
expected_size = 102540417
expected_sha = '11ad3fa62ca79e40addfd354a8ec4b7c75143b3038b8d2a807fbc68deab379ca'
digest = hashlib.sha256(Path(CHECKPOINT).read_bytes()).hexdigest()
assert Path(CHECKPOINT).stat().st_size == expected_size
assert digest == expected_sha, (digest, expected_sha)
print('CHECKPOINT VERIFIED:', digest)

In [ ]:
if Path(CACHE_DIR, 'test.pt').exists():
    raise RuntimeError('Refusing to run with test.pt in the feature cache.')
subprocess.run([
    sys.executable, '-B', 'tools/srq_generalization_m25.py', 'extract-train',
    '--config', CONFIG, '--feature-cache-dir', CACHE_DIR,
    '--root', '/content/cifar100', '--backbone-checkpoint', CHECKPOINT,
    '--device', 'cuda', '--batch-size', '128', '--num-workers', '0',
    '--require-clean-git'
], check=True)
assert Path(CACHE_DIR, 'train.pt').is_file()
assert not Path(CACHE_DIR, 'test.pt').exists()
print('M25 TRAIN CACHE: PASS')

In [ ]:
subprocess.run([
    sys.executable, '-B', 'tools/srq_generalization_m25.py', 'run',
    '--config', CONFIG, '--feature-cache-dir', CACHE_DIR,
    '--output-dir', OUTPUT_DIR, '--device', 'cuda', '--require-clean-git'
], check=True)
result = json.loads(Path(OUTPUT_DIR, 'm25_results.json').read_text())
print('STATUS:', result['status'])
print(json.dumps(result['aggregate'], indent=2))
assert result['status'] == 'PASS_M25_RANPAC_RESNET_CIFAR_TRAIN_ONLY'
assert all(result['gates'].values())

In [ ]:
import shutil
export = shutil.make_archive('/content/srq_generalization_m25_ranpac_resnet_cifar_train_only', 'zip', OUTPUT_DIR)
print('EXPORT:', export)
try:
    from google.colab import files
    files.download(export)
except ImportError:
    print('Download the archive from /content manually.')